# Logistic Regression Assumptions – Solution

**Short name (GitHub):** `LogReg_Assumptions`  
Work the skeleton first. This notebook is the worked key plus commentary.

Numbers below were produced with scikit-learn 1.x, `penalty=None`, `solver='lbfgs'`, `random_state=0` (model section) and `random_state=6` (imbalance section). Slight drift across solvers is normal; signs and metric *order* should match.


## Inline cheat-sheet

| Item | Formula / code |
|------|----------------|
| Encode | `map({'M':1,'B':0})` |
| Independence | `id.nunique()==id.count()` → True on this file |
| 10-EPV | $212/10=21.2$ max features |
| Outlier cut | `fractal_dimension_mean` 99th pct ≈ 0.0854, 6 rows dropped |
| Collinear with radius | `perimeter_mean` (r≈0.998), `area_mean` (r≈0.987) |
| Other pair | `compactness_mean`, `concavity_mean` (r≈0.88) |
| Default 4-feature test (rs=0) | acc≈0.906 · prec≈0.851 · rec≈0.905 · F1≈0.877 · AUC≈0.981 |
| CM @0.5 | TN 98, FP 10, FN 6, TP 57 |
| CM @0.25 | FN drops to 2 (FP rises to 14) |
| CM @0.75 | FN rises to 11 (FP drops to 4) |


## 0. Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score,
)
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

%matplotlib inline
sns.set_style("whitegrid")
np.set_printoptions(precision=4, suppress=True)
print("libraries ready")


## 1. Load and encode

In [ ]:
df = pd.read_csv("data/breast_cancer_data.csv")
df = df.loc[:, ~df.columns.str.contains(r"^Unnamed")]
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0}).astype(int)
print(df.head())
print(df.diagnosis.value_counts())
# B=357, M=212. Positivity rate = 212/569 ≈ 0.3726


## 2. Assumptions I

In [ ]:
print(df.diagnosis.value_counts())
print("n classes:", df.diagnosis.nunique())  # 2 → binary assumption holds


In [ ]:
unique_ids = df.id.nunique() == df.id.count()
print(unique_ids)  # True — one row per patient, independence OK


In [ ]:
max_features = min(df.diagnosis.value_counts()) / 10
print(max_features)  # 21.2


In [ ]:
predictor_all = [
    "radius_mean", "texture_mean", "perimeter_mean", "area_mean",
    "smoothness_mean", "compactness_mean", "concavity_mean",
    "concave points_mean", "symmetry_mean", "fractal_dimension_mean",
]
plt.figure(figsize=(10, 4.2))
sns.boxplot(data=np.log(df[predictor_all] + 0.01).apply(zscore))
plt.xticks(rotation=45, ha="right")
plt.title("Log-z boxplot — fractal_dimension_mean has the longest upper tail")
plt.tight_layout(); plt.show()


In [ ]:
q_hi = df["fractal_dimension_mean"].quantile(0.99)
df_filtered = df[df["fractal_dimension_mean"] < q_hi].copy()
print("q99 =", q_hi, " kept =", len(df_filtered), " dropped =", len(df) - len(df_filtered))
# ≈ 0.0854, 563 kept, 6 dropped


In [ ]:
plt.figure(figsize=(10, 4.2))
sns.boxplot(data=np.log(df_filtered[predictor_all] + 0.01).apply(zscore))
plt.xticks(rotation=45, ha="right")
plt.title("After 99th-percentile filter on fractal_dimension_mean")
plt.tight_layout(); plt.show()
# The fractal tail shortens; radius/area still show a few high points — expected in malignant cases.


## 3. Assumptions II

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
sns.regplot(x="radius_mean", y="diagnosis", data=df, logistic=True, ax=axes[0],
            scatter_kws={"alpha": 0.25, "s": 16})
axes[0].set_title("radius_mean — clear sigmoid")
sns.regplot(x="fractal_dimension_mean", y="diagnosis", data=df, logistic=True, ax=axes[1],
            scatter_kws={"alpha": 0.25, "s": 16})
axes[1].set_title("fractal_dimension_mean — flatter, weaker")
plt.tight_layout(); plt.show()


In [ ]:
x = df[predictor_all]
plt.figure(figsize=(9, 7))
sns.heatmap(x.corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0, annot_kws={"size": 7})
plt.title("Mean-feature correlations")
plt.tight_layout(); plt.show()
# radius_mean ≈ perimeter_mean (0.998) ≈ area_mean (0.987)
correlated_pair = ["compactness_mean", "concavity_mean"]  # r ≈ 0.88
print("correlated_pair:", correlated_pair)


## 4. scikit-learn implementation

In [ ]:
predictor_var = ["radius_mean", "texture_mean", "compactness_mean", "symmetry_mean"]
outcome_var = "diagnosis"
x_train, x_test, y_train, y_test = train_test_split(
    df[predictor_var], df[outcome_var], random_state=0, test_size=0.3
)
log_reg = LogisticRegression(penalty=None, fit_intercept=True, max_iter=4000, solver="lbfgs")
print(log_reg.get_params())


In [ ]:
log_reg.fit(x_train, y_train)
coefficients = log_reg.coef_
intercept = log_reg.intercept_
print("coefficients:", coefficients)
print("intercept:", intercept)
# Typical: radius ~ +1.08, texture ~ +0.29, compactness ~ +31, symmetry ~ +30, intercept ~ −30
# Compactness and symmetry are on a 0–1 scale so their raw coefficients look huge.


In [ ]:
y_pred = log_reg.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f"accuracy\t{accuracy:.4f}")
print(f"precision\t{precision:.4f}")
print(f"recall   \t{recall:.4f}")
print(f"f1       \t{f1:.4f}")
# accuracy ≈ 0.906  precision ≈ 0.851  recall ≈ 0.905  f1 ≈ 0.877
# Precision is the lowest — we pay some extra false alarms to keep recall high, which is the right bias here.


In [ ]:
test_conf_matrix = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["actual no", "actual yes"],
    columns=["predicted no", "predicted yes"],
)
print(test_conf_matrix)
#        pred no  pred yes
# act no      98        10
# act yes      6        57
# Correct: 98+57=155 / 171. Dangerous cell = FN = 6.


## 5. Thresholds

In [ ]:
y_pred_prob = log_reg.predict_proba(x_test)
y_pred_class = (y_pred_prob[:, 1] > 0.5) * 1.0
diff = np.array_equal(y_pred_class, y_pred)
print("same as predict()?", diff)


In [ ]:
print("CM 50%"); print(confusion_matrix(y_test, y_pred_class))
print("CM 25%"); print(confusion_matrix(y_test, (y_pred_prob[:, 1] > 0.25) * 1.0))
print("CM 75%"); print(confusion_matrix(y_test, (y_pred_prob[:, 1] > 0.75) * 1.0))
# 25%: FN 2, FP 14   — catches almost every malignancy, more alarms
# 50%: FN 6, FP 10
# 75%: FN 11, FP 4   — conservative, misses more cancers


In [ ]:
thresh = np.linspace(0, 1, 100)
false_negatives = []
for t in thresh:
    cm = confusion_matrix(y_test, (y_pred_prob[:, 1] > t) * 1.0)
    false_negatives.append(cm[1, 0])
thresh_choice = thresh[np.argmax(np.array(false_negatives) >= 2)]
print("thresh_choice =", thresh_choice)
print("FN at that t =", false_negatives[int(np.argmax(np.array(false_negatives) >= 2))])
# Codecademy checkpoint: lowest t on the grid where FN first hits 2.


## 6. ROC / AUC

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob[:, 1])
plt.figure(figsize=(6.2, 5.6))
plt.plot(fpr, tpr, color="darkorange", label="ROC curve")
idx = list(range(len(thresholds)))[1::4]
for i in idx:
    plt.text(fpr[i], tpr[i], f"{thresholds[i]:.2f}", fontsize=7)
clf = DummyClassifier(strategy="most_frequent", random_state=0)
clf.fit(x_train, y_train)
fpr_d, tpr_d, _ = roc_curve(y_test, clf.predict_proba(x_test)[:, 1])
auc_d = roc_auc_score(y_test, clf.predict_proba(x_test)[:, 1])
plt.plot(fpr_d, tpr_d, color="navy", ls="--", label=f"Dummy most-frequent (AUC={auc_d:.2f})")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC — 4-feature unregularized LR")
plt.grid(True, alpha=0.4); plt.legend(loc="lower right")
plt.show()
# For cancer screening we sit on the *left-upper* part: high TPR, tolerate some FPR.
# That corresponds to a threshold *below* 0.5 (the 0.25 mark).


In [ ]:
roc_auc = roc_auc_score(y_test, y_pred_prob[:, 1])
print("ROC AUC:", roc_auc)  # ≈ 0.981 — strong score-level separation


## 7. Class imbalance

In [ ]:
x_train_u, x_test_u, y_train_u, y_test_u = train_test_split(
    df[predictor_var], df[outcome_var], random_state=6, test_size=0.3
)
print("unstrat train pos", float(y_train_u.mean()), "test pos", float(y_test_u.mean()))
x_train_str, x_test_str, y_train_str, y_test_str = train_test_split(
    df[predictor_var], df[outcome_var], random_state=6, test_size=0.3,
    stratify=df[outcome_var],
)
print("strat   train pos", float(y_train_str.mean()), "test pos", float(y_test_str.mean()))


In [ ]:
str_train_positivity_rate = float(y_train_str.mean())
str_test_positivity_rate = float(y_test_str.mean())
print(str_train_positivity_rate, str_test_positivity_rate)
# Both sit next to the population 0.373; the unstratified split can wander by several points.


In [ ]:
log_reg.fit(x_train_str, y_train_str)
y_pred_s = log_reg.predict(x_test_str)
recall_str = recall_score(y_test_str, y_pred_s)
accuracy_str = accuracy_score(y_test_str, y_pred_s)
print("stratified recall, acc:", recall_str, accuracy_str)

log_reg.fit(x_train_u, y_train_u)
y_pred_u = log_reg.predict(x_test_u)
print("unstrat (rs=6) recall, acc:", recall_score(y_test_u, y_pred_u), accuracy_score(y_test_u, y_pred_u))


In [ ]:
log_reg_bal = LogisticRegression(
    penalty=None, fit_intercept=True, max_iter=4000, solver="lbfgs",
    class_weight="balanced",
)
log_reg_bal.fit(x_train_u, y_train_u)
y_pred_b = log_reg_bal.predict(x_test_u)
recall_bal = recall_score(y_test_u, y_pred_b)
accuracy_bal = accuracy_score(y_test_u, y_pred_b)
print("balanced recall, acc:", recall_bal, accuracy_bal)
# Balanced weights usually lift recall (the minority / clinically costly class)
# and can trim overall accuracy by a point or two — an acceptable trade.


## 8. Alternate code

In [ ]:
try:
    import statsmodels.api as sm
    Xc = sm.add_constant(x_train)
    sm_fit = sm.Logit(y_train, Xc).fit(disp=False)
    print(sm_fit.summary())
    print("sklearn coef", log_reg.coef_, "intercept", log_reg.intercept_)
except Exception as exc:
    print("statsmodels unavailable or failed:", exc)
    print("Fallback: sklearn coefficients already printed above.")


In [ ]:
pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(penalty=None, fit_intercept=True, max_iter=4000),
)
pipe.fit(x_train, y_train)
print("scaled-pipeline acc", accuracy_score(y_test, pipe.predict(x_test)))
print("scaled-pipeline rec", recall_score(y_test, pipe.predict(x_test)))
print("raw-unit sklearn acc", accuracy)
# Predictions match closely. Coefficients of pipe.named_steps['logisticregression']
# are per 1 SD, so compactness no longer looks 30× larger than radius.


In [ ]:
def predict_at(proba, t=0.5):
    return (np.asarray(proba) >= t).astype(int)

for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
    pred = predict_at(y_pred_prob[:, 1], t)
    cm = confusion_matrix(y_test, pred)
    print(f"t={t:.1f}  FN={cm[1,0]:2d}  FP={cm[0,1]:2d}  rec={recall_score(y_test, pred):.3f}")


## 9. More practice

In [ ]:
worst_var = ["radius_worst", "texture_worst", "compactness_worst", "symmetry_worst"]
Xtr_w, Xte_w, ytr_w, yte_w = train_test_split(
    df[worst_var], df[outcome_var], random_state=0, test_size=0.3
)
m_w = LogisticRegression(penalty=None, fit_intercept=True, max_iter=4000)
m_w.fit(Xtr_w, ytr_w)
pw = m_w.predict_proba(Xte_w)[:, 1]
print("worst-features recall", recall_score(yte_w, m_w.predict(Xte_w)))
print("worst-features AUC   ", roc_auc_score(yte_w, pw))
print("mean-features  AUC   ", roc_auc)
# Worst-nucleus measurements are taken at the largest abnormal region and
# typically separate classes *at least* as well as the means.


In [ ]:
loans = pd.read_csv("data/lr_assump_loans.csv")
print(loans.head())
print("positivity", loans.default.mean())
print("max features (10-EPV)", loans.default.value_counts().min() / 10)
Lx = loans[["dti", "utilization", "income"]]
Ly = loans["default"]
Ltr, Lte, lytr, lyte = train_test_split(Lx, Ly, random_state=0, test_size=0.3, stratify=Ly)
m_l = LogisticRegression(penalty=None, fit_intercept=True, max_iter=4000)
m_l.fit(Ltr, lytr)
print("coef", m_l.coef_, "intercept", m_l.intercept_)
lp = m_l.predict_proba(Lte)[:, 1]
print("loan recall", recall_score(lyte, m_l.predict(Lte)),
      "auc", roc_auc_score(lyte, lp))
# threshold for FN <= 3
best_t = None
for t in np.linspace(0.05, 0.9, 18):
    fn = confusion_matrix(lyte, (lp >= t).astype(int))[1, 0]
    fp = confusion_matrix(lyte, (lp >= t).astype(int))[0, 1]
    if fn <= 3:
        best_t = t
        print(f"t={t:.2f} FN={fn} FP={fp}")
print("highest t with FN<=3 (last printed) is the most precise of those options")


In [ ]:
screening = (
    "Population screening: prevalence is low, so a default 0.5 cut will look accurate "
    "while missing rare positives. Drive the threshold down (or use class_weight) and "
    "quote recall + PPV, not accuracy."
)
confirming = (
    "A second-reader / confirmatory test can sit at a higher threshold: false alarms "
    "now cost a real biopsy. Precision and specificity matter more than raw recall."
)
ab_test = (
    "Balanced A/B click-through: classes are closer to 50/50 and both errors cost a "
    "similar amount of attention. Accuracy or AUC is a fair headline metric."
)
print(screening); print(confirming); print(ab_test)


## 10. Simulation

In [ ]:
# --- editable parameters ---
N = 569
N_REPS = 20
NOISE = 0.00
T = 0.50
TEST_SIZE = 0.30
SEED = 0
# ---------------------------
rng = np.random.default_rng(SEED)
rows_def, rows_bal = [], []
pool_X = df[predictor_var].to_numpy()
pool_y = df[outcome_var].to_numpy()
for r in range(N_REPS):
    idx = rng.choice(len(df), size=N, replace=(N > len(df)))
    X = pool_X[idx]; y = pool_y[idx]
    try:
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=r, stratify=y)
    except ValueError:
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=r)
    if NOISE > 0:
        flip = rng.random(len(ytr)) < NOISE
        ytr = ytr.copy(); ytr[flip] = 1 - ytr[flip]
    for tag, cw in (("default", None), ("balanced", "balanced")):
        m = LogisticRegression(penalty=None, fit_intercept=True, max_iter=4000, class_weight=cw)
        m.fit(Xtr, ytr)
        proba = m.predict_proba(Xte)[:, 1]
        pred = (proba >= T).astype(int)
        rec = dict(
            recall=recall_score(yte, pred, zero_division=0),
            accuracy=accuracy_score(yte, pred),
            auc=roc_auc_score(yte, proba) if len(np.unique(yte)) == 2 else np.nan,
        )
        (rows_def if tag == "default" else rows_bal).append(rec)
sim_def = pd.DataFrame(rows_def); sim_bal = pd.DataFrame(rows_bal)
print("DEFAULT\n", sim_def.agg(["mean", "std"]).round(3))
print("BALANCED\n", sim_bal.agg(["mean", "std"]).round(3))
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for ax, col in zip(axes, ["recall", "accuracy", "auc"]):
    ax.boxplot([sim_def[col].dropna(), sim_bal[col].dropna()], labels=["default", "balanced"])
    ax.set_title(col); ax.set_ylim(0.4, 1.02)
fig.suptitle(f"N={N}  T={T}  noise={NOISE}  reps={N_REPS}", y=1.03)
plt.tight_layout(); plt.show()


## 11. Audience rewrite

In [ ]:
expert = (
    "On the rs=0 hold-out (n=171) an unpenalized 4-feature logit attains AUC 0.981. "
    "Operating at t=0.25 rather than 0.50 moves the confusion matrix from (FN=6, FP=10) "
    "to (FN=2, FP=14). Compactness and symmetry dominate the linear predictor because "
    "they are measured on a unit interval; radius (~cm) has a smaller raw coefficient "
    "but remains logit-linear (regplot). Perimeter/area were dropped (r>0.98 with radius)."
)
technician = (
    "Encode M/B as 1/0, drop the empty column, confirm one row per id. Fit "
    "LogisticRegression(penalty=None) on radius_mean, texture_mean, compactness_mean, "
    "symmetry_mean. Call predict_proba and flag a case if column 1 ≥ 0.25 if the SOP "
    "is 'miss at most two cancers on this test list'. Recheck IDs after any 99th-pct filter."
)
executive = (
    "A simple four-measurement model separates benign from malignant well (AUC 0.98). "
    "If we accept about 14 extra false alarms we cut missed cancers on the test list "
    "from six to two. That is the trade the tumour board should pick, not the default "
    "50% cut the software ships with. This is a decision-support score, not a diagnosis."
)
nonspecialist = (
    "Doctors already look at how large and irregular a cell cluster is. The model turns "
    "four of those measurements into a 0–100 risk score. Using a cautious cut-off, it "
    "caught almost every dangerous case in our test file and sent a handful of harmless "
    "ones for a second look — which is what you want when the cost of missing cancer is high."
)
print("EXPERT\n", expert)
print("TECHNICIAN\n", technician)
print("EXECUTIVE\n", executive)
print("NONSPECIALIST\n", nonspecialist)


## 12. Top-10 applications and when not to use this

In [ ]:
applications = [
    "1. Diagnostic classification from lab / imaging features (this notebook).",
    "2. Credit-default / loan-approval probability (PD scorecards).",
    "3. Churn / retention — will this subscriber cancel in 30 days?",
    "4. Email / ad conversion — did the recipient click or buy?",
    "5. Fraud / AML alert triage (with a low threshold and human review).",
    "6. Hospital readmission or ICU-transfer risk within 72 hours.",
    "7. Manufacturing pass/fail from sensor summaries (after linearity checks).",
    "8. Political / survey binary outcomes (voted / did not; support / oppose).",
    "9. A/B experiment analysis when the outcome is a conversion flag.",
    "10. Wildlife / ecology presence-absence models with a small covariate set.",
]
not_appropriate = [
    "1. Target has 3+ unordered classes — use multinomial logit or a proper multiclass model.",
    "2. Observations are repeated on the same person / hospital — independence fails; use GEE or mixed logit.",
    "3. Events-per-variable << 10 in the minority class — MLE is unstable; shrink, reduce features, or gather data.",
    "4. The logit is clearly curved (regplot not sigmoidal) — add splines, GAMs, or a non-linear model.",
    "5. Predictors are almost collinear and you need *interpretable* SEs — drop/combine first or use ridge.",
    "6. You need a causal effect, not a risk score — LR on observational features is not an identification strategy.",
    "7. Prevalence is 0.1% and you only have 20 positives — even balanced weights will not invent information.",
    "8. The decision is a continuous dose / time-to-event — use regression or survival models, not a 0/1 logit.",
]
for row in applications: print(row)
print("--- not appropriate ---")
for row in not_appropriate: print(row)


## 13. Done-when checklist

Matches the skeleton. Saved figures live next to this notebook:

- `lr_assump_flowchart.png`
- `lr_assump_boxplot.png` / `lr_assump_boxplot_filtered.png`
- `lr_assump_logit_curves.png`
- `lr_assump_corr_heatmap.png`
- `lr_assump_proba_hist.png`
- `lr_assump_roc.png`
- `lr_assump_threshold_sweep.png`
- `lr_assump_simulation.png`
